# Assignment 11 ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â Fine-Tuning GPT-2 on a Custom Text Style

**Objective:** Fine-tune a pretrained GPT-2 model on a custom writing style and compare the original and fine-tuned outputs.

**Prompt:** `The scientist opened the door and found`


## 1. Install required packages

In [38]:
%pip install -q torch transformers datasets accelerate

Note: you may need to restart the kernel to use updated packages.


## 2. Imports and configuration

In [39]:
# Imports and configuration for the active Jupyter kernel.

from pathlib import Path

import inspect

import random

import re

import torch
import datasets
import transformers

from torch.utils.data import Dataset

from transformers import (

    GPT2LMHeadModel,

    GPT2TokenizerFast,

    TrainingArguments,

    Trainer,

    DataCollatorForLanguageModeling,

)



ROOT = Path.cwd()

MODEL_NAME = "gpt2"

PROMPT = "The scientist opened the door and found"

DATASET_PATH = ROOT / "custom_style_dataset.txt"

BASE_OUTPUT = ROOT / "base_model_output.txt"

FINE_OUTPUT = ROOT / "fine_tuned_model_output.txt"

COMPARISON = ROOT / "comparison.txt"

MODEL_PATH = ROOT / "gpt2-finetuned-custom-style"



# PyTorch uses CPU when CUDA is unavailable; this same device is used for generation.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Transformers version:", transformers.__version__)

print("PyTorch version:", torch.__version__)

print("Device:", device)

Transformers version: 5.19.0
PyTorch version: 2.14.1+cpu
Device: cpu


## 3. Create the 250-line custom narrative dataset

In [40]:
# Build 250 distinct, clean narrative lines with a consistent restrained style.

random.seed(11)

locations = ["the quiet laboratory", "the observatory", "the old archive", "the narrow control room", "the coastal station", "the stone greenhouse", "the hilltop radio room", "the winter field cabin"]

sensory_details = ["a faint ticking clock", "salt in the air", "a dim glass window", "a row of silent instruments", "a folded map", "dust on the brass handle", "rain tapping the roof", "a pale mark on the floor", "the smell of wet paper", "a low electrical hum"]

actions = ["paused to listen", "checked the instruments", "watched the rain", "held the notebook tightly", "looked toward the corridor", "counted the measured pulses", "turned the small brass key", "wrote down the time", "lifted the lantern", "followed the narrow beam"]

findings = ["a second set of footprints", "a message in the margin", "a warm cup beside the chart", "a signal returning after midnight", "a drawer left slightly open", "a name scratched into the wood", "one instrument still moving", "a fresh line across the dust", "a sealed envelope without a stamp", "a light beyond the far window"]



lines = []

for i in range(250):

    location = locations[i % len(locations)]

    detail = sensory_details[(i * 3 + i // len(locations)) % len(sensory_details)]

    action = actions[(i * 7 + i // len(actions)) % len(actions)]

    finding = findings[(i * 9 + i // len(findings)) % len(findings)]

    lines.append(

        f"At the edge of {location}, the scientist {action}. {detail.capitalize()} "

        f"rested nearby. The scientist marked observation {i + 1} in the log and waited for morning. Behind the door, the scientist found {finding}. "

        "A careful note was added before the room fell quiet again."

    )



assert len(lines) == 250

assert all(line.strip() and "\n" not in line for line in lines)

assert len(set(lines)) == 250, "Narrative lines must be distinct."

DATASET_PATH.write_text("\n".join(lines), encoding="utf-8")

print(f"Saved exactly {len(lines)} distinct narrative lines to {DATASET_PATH}")

Saved exactly 250 distinct narrative lines to c:\Users\Lenovo\Downloads\custom_style_dataset.txt


## 4. Load GPT-2 and generate the original output

In [41]:
# Load the pretrained base model and save its genuine generated continuation.
tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
model.config.pad_token_id = tokenizer.eos_token_id
model.to(device)
model.eval()

generation_inputs = tokenizer(PROMPT, return_tensors="pt").to(device)
with torch.no_grad():
    generated = model.generate(
        **generation_inputs,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )
base_text = tokenizer.decode(generated[0], skip_special_tokens=True)
BASE_OUTPUT.write_text(base_text, encoding="utf-8")
print("Saved base GPT-2 output to:", BASE_OUTPUT)
print(base_text)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Saved base GPT-2 output to: c:\Users\Lenovo\Downloads\base_model_output.txt
The scientist opened the door and found a large, wooden table with a large wooden bowl sitting on the bottom of the table. The scientist was wearing a red and white suit, matching that of the man who brought him to work. He wore the suit as a shield and a hat.

"So you have been using a knife


## 5. Prepare the dataset for causal language modeling

In [42]:
# Tokenize the prepared text and split it into fixed-length causal language-model blocks.
clean_lines = [line.strip() for line in DATASET_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
assert len(clean_lines) == 250, f"Expected 250 lines; found {len(clean_lines)}."

block_size = 64
all_token_ids = []
for line in clean_lines:
    all_token_ids.extend(tokenizer.encode(line, add_special_tokens=False))
    all_token_ids.append(tokenizer.eos_token_id)
blocks = [all_token_ids[i:i + block_size] for i in range(0, len(all_token_ids) - block_size + 1, block_size)]

class NarrativeBlocks(Dataset):
    """Small PyTorch dataset that avoids an extra datasets package dependency."""
    def __init__(self, token_blocks):
        self.token_blocks = token_blocks
    def __len__(self):
        return len(self.token_blocks)
    def __getitem__(self, index):
        return {"input_ids": self.token_blocks[index]}

lm_dataset = NarrativeBlocks(blocks)
assert len(lm_dataset) > 0, "Tokenization produced no training blocks."
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print("Training blocks:", len(lm_dataset))

Training blocks: 242


## 6. Fine-tune GPT-2

In [43]:
# Build only arguments supported by the installed Transformers version.
training_values = {
    "output_dir": str(ROOT / "gpt2-training-runs"),
    "per_device_train_batch_size": 1,
    "max_steps": 20,
    "logging_steps": 1,
    "report_to": "none",
    "save_strategy": "no",
    "data_seed": 11,
    "use_cpu": device.type == "cpu",
}
training_supported = inspect.signature(TrainingArguments.__init__).parameters
training_args = TrainingArguments(**{
    name: value for name, value in training_values.items()
    if name in training_supported
})

model_for_training = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
model_for_training.config.pad_token_id = tokenizer.eos_token_id
trainer_values = {
    "model": model_for_training,
    "args": training_args,
    "train_dataset": lm_dataset,
    "data_collator": data_collator,
}
trainer_supported = inspect.signature(Trainer.__init__).parameters
# Transformers versions supporting processing_class use it; older versions use tokenizer.
if "processing_class" in trainer_supported:
    trainer_values["processing_class"] = tokenizer
elif "tokenizer" in trainer_supported:
    trainer_values["tokenizer"] = tokenizer
trainer = Trainer(**{
    name: value for name, value in trainer_values.items()
    if name in trainer_supported
})

trainer.train()
trainer.save_model(str(MODEL_PATH))
tokenizer.save_pretrained(str(MODEL_PATH))
print("Fine-tuned model saved to:", MODEL_PATH)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Step,Training Loss
1,4.881379
2,4.194762
3,4.102804
4,3.856510
5,3.835638
6,3.344490
7,3.139600
8,2.909194
9,2.855936
10,2.333730


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved to: c:\Users\Lenovo\Downloads\gpt2-finetuned-custom-style


## 7. Generate output from the fine-tuned model

In [44]:
# Load the saved fine-tuned model and generate its genuine continuation.
fine_model = GPT2LMHeadModel.from_pretrained(MODEL_PATH).to(device)
fine_model.eval()
fine_inputs = tokenizer(PROMPT, return_tensors="pt").to(device)
with torch.no_grad():
    fine_generated = fine_model.generate(
        **fine_inputs,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )
fine_text = tokenizer.decode(fine_generated[0], skip_special_tokens=True)
FINE_OUTPUT.write_text(fine_text, encoding="utf-8")
print("Saved fine-tuned GPT-2 output to:", FINE_OUTPUT)
print(fine_text)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Saved fine-tuned GPT-2 output to: c:\Users\Lenovo\Downloads\fine_tuned_model_output.txt
The scientist opened the door and found a small cup nearby.

"I must report to you again. You have not seen me in days," said the scientist.

"Thank you again for your time," replied the scientist.

The scientist waited in the quiet room for morning. Behind the door, the scientist heard


## 8. Before-and-after comparison

In [45]:
# Compare observable style features in the two real generated texts.

def style_features(text):

    words = re.findall(r"\b[\w']+\b", text.lower())

    sentences = [part for part in re.split(r"[.!?]+", text) if part.strip()]

    sensory_words = {"quiet", "dim", "faint", "silent", "cold", "warm", "dark", "pale", "dust", "rain", "smell", "sound", "light", "shadow", "window", "room", "door", "laboratory"}

    return {

        "word_count": len(words),

        "unique_word_count": len(set(words)),

        "sentence_count": len(sentences),

        "sensory_or_scene_word_count": sum(word in sensory_words for word in words),

        "dialogue_mark_count": text.count('"') + text.count("Ã¢â‚¬Å“") + text.count("Ã¢â‚¬â€"),

    }



base_features = style_features(base_text)

fine_features = style_features(fine_text)



def difference_line(label, key, unit=""):

    original = base_features[key]

    tuned = fine_features[key]

    direction = "higher" if tuned > original else "lower" if tuned < original else "the same"

    return (f"{label}: original GPT-2 = {original}{unit}; fine-tuned GPT-2 = {tuned}{unit} "

            f"(fine-tuned value is {direction}).")



style_differences = [

    difference_line("Output length", "word_count", " words"),

    difference_line("Vocabulary variety", "unique_word_count", " unique words"),

    difference_line("Sensory and scene vocabulary", "sensory_or_scene_word_count", " occurrences"),

    difference_line("Sentence count", "sentence_count", " sentences"),

]

comparison = (

    "Prompt:\n" + PROMPT + "\n\n"

    "Original GPT-2 output:\n" + base_text + "\n\n"

    "Fine-tuned GPT-2 output:\n" + fine_text + "\n\n"

    "Writing-style differences (measured from these outputs):\n"

    "1. " + style_differences[0] + "\n"

    "2. " + style_differences[1] + "\n"

    "3. " + style_differences[2] + "\n"

    "4. " + style_differences[3] + "\n"

)

COMPARISON.write_text(comparison, encoding="utf-8")

print(comparison)

Prompt:
The scientist opened the door and found

Original GPT-2 output:
The scientist opened the door and found a large, wooden table with a large wooden bowl sitting on the bottom of the table. The scientist was wearing a red and white suit, matching that of the man who brought him to work. He wore the suit as a shield and a hat.

"So you have been using a knife

Fine-tuned GPT-2 output:
The scientist opened the door and found a small cup nearby.

"I must report to you again. You have not seen me in days," said the scientist.

"Thank you again for your time," replied the scientist.

The scientist waited in the quiet room for morning. Behind the door, the scientist heard

Writing-style differences (measured from these outputs):
1. Output length: original GPT-2 = 59 words; fine-tuned GPT-2 = 51 words (fine-tuned value is lower).
2. Vocabulary variety: original GPT-2 = 40 unique words; fine-tuned GPT-2 = 34 unique words (fine-tuned value is lower).
3. Sensory and scene vocabulary: origin

## 9. Assignment outputs

The notebook creates:

- `custom_style_dataset.txt`
- `base_model_output.txt`
- `fine_tuned_model_output.txt`
- `comparison.txt`
- `gpt2-finetuned-custom-style/`
